# Wilcoxon Significance testing

## Set Up

- Note that need to ensure assumptions met:
    - Distribution of differences between paired scores is ~symmetric around median
- Note that each compared "model" here is a different configuration (agg type, alpha val, llama model, etc)

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.utils import ALPHA_LIST
import pandas as pd
from statsmodels.stats.multitest import multipletests
from scipy.stats import wilcoxon
from sklearn.metrics import r2_score, root_mean_squared_error
import matplotlib.pyplot as plt
import numpy as np

pd.set_option("display.max_columns", 50)

In [ ]:
def root_squared_error(y_true, y_pred):
    diff = y_true - y_pred
    if diff.all() == 0:
        return 0
    else:
        return np.sqrt(diff**2)


def squared_error(y_true, y_pred):
    diff = y_true - y_pred
    return diff**2


def export_hypothesis(
    impact_type, table, hypothesis_num, export_dir=BASE_PATH / "sig_tests"
):
    file_path = export_dir / f"hypothesis_{hypothesis_num}" / f"{impact_type}.tsv"
    file_path.parent.mkdir(exist_ok=True, parents=True)
    if file_path.exists():
        raise ValueError(f"Table already exists at path {file_path}. Aborting...")
    table.to_csv(file_path, sep="\t")

## Main Hypotheses

### 2T vs 2S


In [ ]:
def get_hypothesis_1(impact_type):
    """
    Given a two-tier modeling schema:
        H_0: text level error >= sum level
        H_A: text level error < sum level
    """
    loop_dir = BASE_PATH / "old_preds/baseline/2_layers/ridge"
    sig_model_list = []
    for agg_dir in loop_dir.iterdir():
        agg_name = agg_dir.name
        impact_dir = agg_dir / impact_type
        for llama_dir in impact_dir.iterdir():
            llama_name = llama_dir.name
            for desc_seg_dir in llama_dir.iterdir():
                desc_seg_name = desc_seg_dir.name
                for alpha_dir in desc_seg_dir.iterdir():
                    alpha_dir_split = alpha_dir.name.split("_")
                    alpha3 = float(alpha_dir_split[1])
                    ##### At this point, we have all the unique params of this model
                    ## Now aggregate folds to get RSE for each participant in a given repeat, and avg over all repeats
                    ## End with avg RSE for each participant over repeats
                    repeat_dfs = []
                    for repeat_dir in alpha_dir.iterdir():
                        split_dfs = []
                        for split_path in repeat_dir.iterdir():
                            split_file = pd.read_parquet(split_path)
                            split_dfs.append(split_file)
                        combined_splits_df = pd.concat(split_dfs)
                        combined_splits_df["rse_2S"] = squared_error(
                            y_true=combined_splits_df["true"],
                            y_pred=combined_splits_df["pred_stage1"],
                        )
                        combined_splits_df["rse_2T"] = squared_error(
                            y_true=combined_splits_df["true"],
                            y_pred=combined_splits_df["pred_stage3_total"],
                        )
                        repeat_dfs.append(
                            combined_splits_df[
                                [
                                    "DAWBA_ID",
                                    # "true",
                                    # "pred_stage1",
                                    # "pred_stage3_total",
                                    "rse_2S",
                                    "rse_2T",
                                ]
                            ].copy()
                        )
                    all_preds = pd.concat(repeat_dfs, ignore_index=True)
                    ## Aggregate RSE over n repeats
                    participant_avg = all_preds.groupby("DAWBA_ID", as_index=False).agg(
                        mrse_2S=("rse_2S", "mean"),
                        mrse_2T=("rse_2T", "mean"),
                        n_repeats=("rse_2S", "size"),
                    )
                    assert (
                        participant_avg["n_repeats"] == 10
                    ).all(), "Some participants have missing repeats!"
                    diff = participant_avg["mrse_2T"] - participant_avg["mrse_2S"]
                    stat, p_val = wilcoxon(x=diff, alternative="less")
                    sig_model_list.append(
                        {
                            "avg_diff": round(np.mean(diff), 4),
                            "p_val": p_val,
                            "stat": stat,
                            "agg": agg_name,
                            "llama_model": llama_name,
                            "desc_seg": desc_seg_name,
                            "alpha": alpha3,
                        }
                    )
                    # median = diff.median()
                    # diff.hist(bins=20)
                    # plt.axvline(median, linestyle='--',color='red')  # vertical line at median
                    # plt.show()
    sig_model_df = pd.DataFrame(sig_model_list)
    _, sig_model_df["p_val_correct"], _, _ = multipletests(
        pvals=sig_model_df["p_val"], method="fdr_bh"
    )
    return sig_model_df

In [ ]:
self_hypothesis_1 = get_hypothesis_1("self")
display(self_hypothesis_1[self_hypothesis_1["p_val_correct"] < 0.05])

parent_hypothesis_1 = get_hypothesis_1("parent")
display(parent_hypothesis_1[parent_hypothesis_1["p_val_correct"] < 0.05])

In [ ]:
export_hypothesis(impact_type="self", table=self_hypothesis_1, hypothesis_num=1)
export_hypothesis(impact_type="parent", table=parent_hypothesis_1, hypothesis_num=1)

### 3T vs 3Q

In [ ]:
def get_hypothesis_2(impact_type):
    """
    Given a three-tier modeling schema:
        H_0: text level error >= questionnaire level
        H_A: text level < questionnaire level
    """
    loop_dir = BASE_PATH / "preds/baseline/3_layers/ridge"
    sig_model_list = []
    for agg_dir in loop_dir.iterdir():
        agg_name = agg_dir.name
        impact_dir = agg_dir / impact_type
        for llama_dir in impact_dir.iterdir():
            llama_name = llama_dir.name
            for desc_seg_dir in llama_dir.iterdir():
                desc_seg_name = desc_seg_dir.name
                for alpha_dir in desc_seg_dir.iterdir():
                    alpha_dir_split = alpha_dir.name.split("_")
                    alpha2 = float(alpha_dir_split[1])
                    alpha3 = float(alpha_dir_split[2])
                    ##### At this point, we have all the unique params of this model
                    ## Now aggregate folds to get RSE for each participant in a given repeat, and avg over all repeats
                    ## End with avg RSE for each participant over repeats
                    repeat_dfs = []
                    for repeat_dir in alpha_dir.iterdir():
                        split_dfs = []
                        for split_path in repeat_dir.iterdir():
                            split_file = pd.read_parquet(split_path)
                            split_dfs.append(split_file)
                        combined_splits_df = pd.concat(split_dfs)
                        combined_splits_df["rse_3Q"] = squared_error(
                            y_true=combined_splits_df["true"],
                            y_pred=combined_splits_df["pred_stage2_total"],
                        )
                        combined_splits_df["rse_3T"] = squared_error(
                            y_true=combined_splits_df["true"],
                            y_pred=combined_splits_df["pred_stage3_total"],
                        )
                        repeat_dfs.append(
                            combined_splits_df[
                                [
                                    "DAWBA_ID",
                                    # "true",
                                    # "pred_stage1",
                                    # "pred_stage3_total",
                                    "rse_3Q",
                                    "rse_3T",
                                ]
                            ].copy()
                        )
                    all_preds = pd.concat(repeat_dfs, ignore_index=True)
                    # avg RSE over n repeats for each participant
                    participant_avg = all_preds.groupby("DAWBA_ID", as_index=False).agg(
                        mrse_3Q=("rse_3Q", "mean"),
                        mrse_3T=("rse_3T", "mean"),
                        n_repeats=("rse_3Q", "size"),
                    )
                    assert (
                        participant_avg["n_repeats"] == 10
                    ).all(), "Some participants have missing repeats!"
                    diff = participant_avg["mrse_3T"] - participant_avg["mrse_3Q"]
                    stat, p_val = wilcoxon(x=diff, alternative="less")
                    sig_model_list.append(
                        {
                            "agg": agg_name,
                            "llama_model": llama_name,
                            "desc_seg": desc_seg_name,
                            "alpha2": alpha2,
                            "alpha3": alpha3,
                            "p_val": p_val,
                            "stat": stat,
                            "avg_diff": round(np.mean(diff), 4),
                        }
                    )
                    # median = diff.median()
                    # diff.hist(bins=20)
                    # plt.axvline(median, linestyle='--',color='red')  # vertical line at median
                    # plt.show()
    sig_model_df = pd.DataFrame(sig_model_list)
    _, sig_model_df["p_val_correct"], _, _ = multipletests(
        pvals=sig_model_df["p_val"], method="fdr_bh"
    )
    return sig_model_df

In [ ]:
self_hypothesis_2 = get_hypothesis_2("self")
display(self_hypothesis_2[self_hypothesis_2["p_val_correct"] < 0.05])

parent_hypothesis_2 = get_hypothesis_2("parent")
display(parent_hypothesis_2[parent_hypothesis_2["p_val_correct"] < 0.05])

In [ ]:
export_hypothesis(impact_type="self", table=self_hypothesis_2, hypothesis_num=2)
export_hypothesis(impact_type="parent", table=parent_hypothesis_2, hypothesis_num=2)

### 2T vs 3Q

In [ ]:
def get_hypothesis_3(impact_type, param_dict, base_dir):
    """
    Given a tw0 and three-tier modeling schema:
        H_0: 2-level text error >= 3-level questionnaire level
        H_A: 2-level text < 3-level questionnaire level
    """
    # loop_dir = BASE_PATH / "preds/baseline/3_layers/ridge"
    sig_model_list = []
    for agg_type in param_dict["agg"]:
        for llama_model in param_dict["llama_model"]:
            if agg_type in ["avg", "max", "block"]:
                desc_seg_list = param_dict["seg"]
            else:
                desc_seg_list = param_dict["desc"]
            for desc_seg in desc_seg_list:
                for alpha in param_dict["alpha_2"]:
                    alpha_dir_2 = (
                        base_dir
                        / "2_layers/ridge"
                        / agg_type
                        / impact_type
                        / llama_model
                        / desc_seg
                        / f"alpha_{alpha}"
                    )
                    # use arbitrary val for second value of alpha, since only first is used in questionnaire level
                    alpha_dir_3 = (
                        base_dir
                        / "3_layers/ridge"
                        / agg_type
                        / impact_type
                        / llama_model
                        / desc_seg
                        / f"alpha_{alpha}_0.5"
                    )
                    ##### At this point, we have all the unique params of this model
                    ## Now aggregate folds to get RSE for each participant in a given repeat, and avg over all repeats
                    ## End with avg RSE for each participant over repeats
                    repeat_dfs = []
                    for repeat in param_dict["repeats"]:
                        split_dfs_2 = []
                        split_dfs_3 = []
                        for split_name in param_dict["splits"]:
                            split_path_2 = alpha_dir_2 / repeat / split_name
                            split_path_3 = alpha_dir_3 / repeat / split_name
                            split_file_2 = pd.read_parquet(split_path_2)
                            split_file_3 = pd.read_parquet(split_path_3)
                            split_dfs_2.append(split_file_2)
                            split_dfs_3.append(split_file_3)
                        combined_splits_df_2 = pd.concat(split_dfs_2)
                        combined_splits_df_3 = pd.concat(split_dfs_3)
                        combined_splits_df_2["rse_2T"] = squared_error(
                            y_true=combined_splits_df_2["true"],
                            y_pred=combined_splits_df_2["pred_stage3_total"],
                        )
                        combined_splits_df_3["rse_3Q"] = squared_error(
                            y_true=combined_splits_df_3["true"],
                            y_pred=combined_splits_df_3["pred_stage2_total"],
                        )
                        combined_all = pd.merge(
                            combined_splits_df_2[["DAWBA_ID", "rse_2T"]],
                            combined_splits_df_3[["DAWBA_ID", "rse_3Q"]],
                            on="DAWBA_ID",
                        )
                        repeat_dfs.append(combined_all.copy())
                    all_preds = pd.concat(repeat_dfs, ignore_index=True)
                    # avg RSE over n repeats for each participant
                    participant_avg = all_preds.groupby("DAWBA_ID", as_index=False).agg(
                        mrse_3Q=("rse_3Q", "mean"),
                        mrse_2T=("rse_2T", "mean"),
                        n_repeats=("rse_3Q", "size"),
                    )
                    assert (
                        participant_avg["n_repeats"] == 10
                    ).all(), "Some participants have missing repeats!"
                    diff = participant_avg["mrse_2T"] - participant_avg["mrse_3Q"]
                    stat, p_val = wilcoxon(x=diff, alternative="less")
                    sig_model_list.append(
                        {
                            "agg": agg_type,
                            "llama_model": llama_model,
                            "desc_seg": desc_seg,
                            "alpha2": alpha,
                            "p_val": p_val,
                            "stat": stat,
                            "avg_diff": round(np.mean(diff), 4),
                            "std_diff": round(np.std(diff), 4),
                            "n": len(diff),
                        }
                    )
                    # median = diff.median()
                    # diff.hist(bins=20)
                    # plt.axvline(median, linestyle='--',color='red')  # vertical line at median
                    # plt.show()
    sig_model_df = pd.DataFrame(sig_model_list)
    _, sig_model_df["p_val_correct"], _, _ = multipletests(
        pvals=sig_model_df["p_val"], method="fdr_bh"
    )
    return sig_model_df

In [ ]:
param_dict = {
    "agg": ["0_shot_binary", "0_shot_proba-limited", "avg", "block", "max"],
    "llama_model": ["Llama-3.3-8B-Instruct", "Llama-3.3-70B-Instruct"],
    "desc": ["desc_augment", "desc_augment_neg", "desc_original"],
    "seg": ["pt_noshort", "turns"],
    "alpha_2": ALPHA_LIST,
    "repeats": [f"repeat_{i}" for i in range(10)],
    "splits": [f"split_{i}.parquet" for i in range(3)],
}
self_hypothesis_3 = get_hypothesis_3(
    impact_type="self", param_dict=param_dict, base_dir=BASE_PATH / "preds" / "baseline"
)
display(self_hypothesis_3[self_hypothesis_3["p_val_correct"] < 0.05])
parent_hypothesis_3 = get_hypothesis_3(
    impact_type="parent",
    param_dict=param_dict,
    base_dir=BASE_PATH / "preds" / "baseline",
)
display(parent_hypothesis_3[parent_hypothesis_3["p_val_correct"] < 0.05])

In [ ]:
export_hypothesis(impact_type="self", table=self_hypothesis_3, hypothesis_num=3)
export_hypothesis(impact_type="parent", table=parent_hypothesis_3, hypothesis_num=3)

### Intermediate Intersect hypotheses

Parent

In [ ]:
import_path = BASE_PATH / "sig_tests"
parent_hypothesis_1 = pd.read_csv(
    import_path / "hypothesis_1/parent.tsv", sep="\t", index_col=0
)
parent_hypothesis_2 = pd.read_csv(
    import_path / "hypothesis_2/parent.tsv", sep="\t", index_col=0
)
parent_hypothesis_3 = pd.read_csv(
    import_path / "hypothesis_3/parent.tsv", sep="\t", index_col=0
)
# parent_hypothesis_4 = pd.read_csv(import_path / "hypothesis_4/parent.tsv", sep='\t', index_col=0)
# parent_hypothesis_5 = pd.read_csv(import_path / "hypothesis_5/parent.tsv", sep='\t', index_col=0)

In [ ]:
ph1_mcsig = parent_hypothesis_1.query("p_val_correct < 0.05")
ph1_mcsig["alpha2"] = ph1_mcsig["alpha"]
ph2_mcsig = parent_hypothesis_2.query("p_val_correct < 0.05")
ph12_mcsig = ph1_mcsig.merge(
    ph2_mcsig, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
)
# if "alpha" in parent_hypothesis_3.iloc[0]['alpha2']:
#     parent_hypothesis_3['alpha2'] = parent_hypothesis_3['alpha2'].str.split('_').str[-1].astype(float)
ph123 = ph12_mcsig.merge(
    parent_hypothesis_3, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
).sort_values("p_val")
ph123 = (
    ph123.sort_values(["agg", "llama_model", "desc_seg", "alpha2", "alpha3"])
    .groupby(["agg", "llama_model", "desc_seg", "alpha2"])
    .first()
    .reset_index()
)
_, ph123["p_val_smaller_compare"], _, _ = multipletests(ph123.p_val, method="fdr_bh")
ph123[ph123["p_val_smaller_compare"] < 0.05]

Self

In [ ]:
import_path = BASE_PATH / "sig_tests"
self_hypothesis_1 = pd.read_csv(
    import_path / "hypothesis_1/self.tsv", sep="\t", index_col=0
)
self_hypothesis_2 = pd.read_csv(
    import_path / "hypothesis_2/self.tsv", sep="\t", index_col=0
)
self_hypothesis_3 = pd.read_csv(
    import_path / "hypothesis_3/self.tsv", sep="\t", index_col=0
)
# self_hypothesis_4 = pd.read_csv(import_path / "hypothesis_4/self.tsv", sep='\t', index_col=0)
# self_hypothesis_5 = pd.read_csv(import_path / "hypothesis_5/self.tsv", sep='\t', index_col=0)

In [ ]:
sh1_mcsig = self_hypothesis_1.query("p_val_correct < 0.05")
sh1_mcsig["alpha2"] = sh1_mcsig["alpha"]
sh2_mcsig = self_hypothesis_2.query("p_val_correct < 0.05")
sh12_mcsig = sh1_mcsig.merge(
    sh2_mcsig, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
)
# if "alpha" in self_hypothesis_3.iloc[0]['alpha2']:
#     self_hypothesis_3['alpha2'] = self_hypothesis_3['alpha2'].str.split('_').str[-1].astype(float)
sh123 = sh12_mcsig.merge(
    self_hypothesis_3, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
).sort_values("p_val")
sh123 = (
    sh123.sort_values(["agg", "llama_model", "desc_seg", "alpha2", "alpha3"])
    .groupby(["agg", "llama_model", "desc_seg", "alpha2"])
    .first()
    .reset_index()
)
_, sh123["p_val_smaller_compare"], _, _ = multipletests(sh123.p_val, method="fdr_bh")
sh123[sh123["p_val_smaller_compare"] < 0.05]

### 1S vs 1T

In [ ]:
def get_hypothesis_4(impact_type):
    """
    Given a one-tier modeling schema (each feature type models raw impact):
        H_0: text error >= sum
        H_A: text < sum
    """
    loop_dir = BASE_PATH / "preds/baseline/1_layers/ridge"
    sig_model_list = []
    for agg_dir in loop_dir.iterdir():
        agg_name = agg_dir.name
        impact_dir = agg_dir / impact_type
        for llama_dir in impact_dir.iterdir():
            llama_name = llama_dir.name
            for desc_seg_dir in llama_dir.iterdir():
                desc_seg_name = desc_seg_dir.name
                for alpha_dir in desc_seg_dir.iterdir():
                    alpha_dir_split = alpha_dir.name.split("_")
                    alpha3 = float(alpha_dir_split[1])
                    ##### At this point, we have all the unique params of this model
                    ## Now aggregate folds to get RSE for each participant in a given repeat, and avg over all repeats
                    ## End with avg RSE for each participant over repeats
                    repeat_dfs = []
                    for repeat_dir in alpha_dir.iterdir():
                        split_dfs = []
                        for split_path in repeat_dir.iterdir():
                            split_file = pd.read_parquet(split_path)
                            split_dfs.append(split_file)
                        combined_splits_df = pd.concat(split_dfs)
                        combined_splits_df["rse_1S"] = squared_error(
                            y_true=combined_splits_df["true"],
                            y_pred=combined_splits_df["pred_stage1"],
                        )
                        combined_splits_df["rse_1T"] = squared_error(
                            y_true=combined_splits_df["true"],
                            y_pred=combined_splits_df["pred_stage3_total"],
                        )
                        repeat_dfs.append(
                            combined_splits_df[
                                [
                                    "DAWBA_ID",
                                    "rse_1S",
                                    "rse_1T",
                                ]
                            ].copy()
                        )
                    all_preds = pd.concat(repeat_dfs, ignore_index=True)
                    ## Aggregate RSE over n repeats
                    participant_avg = all_preds.groupby("DAWBA_ID", as_index=False).agg(
                        mrse_1S=("rse_1S", "mean"),
                        mrse_1T=("rse_1T", "mean"),
                        n_repeats=("rse_1S", "size"),
                    )
                    assert (
                        participant_avg["n_repeats"] == 10
                    ).all(), "Some participants have missing repeats!"
                    diff = participant_avg["mrse_1T"] - participant_avg["mrse_1S"]
                    stat, p_val = wilcoxon(x=diff, alternative="less")
                    sig_model_list.append(
                        {
                            "avg_diff": round(np.mean(diff), 4),
                            "p_val": p_val,
                            "stat": stat,
                            "agg": agg_name,
                            "llama_model": llama_name,
                            "desc_seg": desc_seg_name,
                            "alpha": alpha3,
                        }
                    )
                    # median = diff.median()
                    # diff.hist(bins=20)
                    # plt.axvline(median, linestyle='--',color='red')  # vertical line at median
                    # plt.show()
    sig_model_df = pd.DataFrame(sig_model_list)
    _, sig_model_df["p_val_correct"], _, _ = multipletests(
        pvals=sig_model_df["p_val"], method="fdr_bh"
    )
    return sig_model_df

In [ ]:
self_hypothesis_4 = get_hypothesis_4("self")
display(self_hypothesis_4[self_hypothesis_4["p_val_correct"] < 0.05])

parent_hypothesis_4 = get_hypothesis_4("parent")
display(parent_hypothesis_4[parent_hypothesis_4["p_val_correct"] < 0.05])

In [ ]:
export_hypothesis(impact_type="self", table=self_hypothesis_4, hypothesis_num=4)
export_hypothesis(impact_type="parent", table=parent_hypothesis_4, hypothesis_num=4)

### 1T vs 3Q

In [ ]:
def get_hypothesis_5(impact_type, param_dict, base_dir):
    """
    Given a 1 and three-tier modeling schema:
        H_0: 1-level text error >= 3-level questionnaire level
        H_A: 1-level text < 3-level questionnaire level
    """
    # loop_dir = BASE_PATH / "preds/baseline/3_layers/ridge"
    sig_model_list = []
    for agg_type in param_dict["agg"]:
        for llama_model in param_dict["llama_model"]:
            if agg_type in ["avg", "max", "block"]:
                desc_seg_list = param_dict["seg"]
            else:
                desc_seg_list = param_dict["desc"]
            for desc_seg in desc_seg_list:
                for alpha in param_dict["alpha_2"]:
                    alpha_dir_2 = (
                        base_dir
                        / "1_layers/ridge"
                        / agg_type
                        / impact_type
                        / llama_model
                        / desc_seg
                        / f"alpha_{alpha}"
                    )
                    # use arbitrary val for second value of alpha, since only first is used in questionnaire level
                    alpha_dir_3 = (
                        base_dir
                        / "3_layers/ridge"
                        / agg_type
                        / impact_type
                        / llama_model
                        / desc_seg
                        / f"alpha_{alpha}_0.5"
                    )
                    ##### At this point, we have all the unique params of this model
                    ## Now aggregate folds to get RSE for each participant in a given repeat, and avg over all repeats
                    ## End with avg RSE for each participant over repeats
                    repeat_dfs = []
                    for repeat in param_dict["repeats"]:
                        split_dfs_2 = []
                        split_dfs_3 = []
                        for split_name in param_dict["splits"]:
                            split_path_2 = alpha_dir_2 / repeat / split_name
                            split_path_3 = alpha_dir_3 / repeat / split_name
                            split_file_2 = pd.read_parquet(split_path_2)
                            split_file_3 = pd.read_parquet(split_path_3)
                            split_dfs_2.append(split_file_2)
                            split_dfs_3.append(split_file_3)
                        combined_splits_df_2 = pd.concat(split_dfs_2)
                        combined_splits_df_3 = pd.concat(split_dfs_3)
                        combined_splits_df_2["rse_1T"] = squared_error(
                            y_true=combined_splits_df_2["true"],
                            y_pred=combined_splits_df_2["pred_stage3_total"],
                        )
                        combined_splits_df_3["rse_3Q"] = squared_error(
                            y_true=combined_splits_df_3["true"],
                            y_pred=combined_splits_df_3["pred_stage2_total"],
                        )
                        combined_all = pd.merge(
                            combined_splits_df_2[["DAWBA_ID", "rse_1T"]],
                            combined_splits_df_3[["DAWBA_ID", "rse_3Q"]],
                            on="DAWBA_ID",
                        )
                        repeat_dfs.append(combined_all.copy())
                    all_preds = pd.concat(repeat_dfs, ignore_index=True)
                    # avg RSE over n repeats for each participant
                    participant_avg = all_preds.groupby("DAWBA_ID", as_index=False).agg(
                        mrse_3Q=("rse_3Q", "mean"),
                        mrse_1T=("rse_1T", "mean"),
                        n_repeats=("rse_3Q", "size"),
                    )
                    assert (
                        participant_avg["n_repeats"] == 10
                    ).all(), "Some participants have missing repeats!"
                    diff = participant_avg["mrse_1T"] - participant_avg["mrse_3Q"]
                    stat, p_val = wilcoxon(x=diff, alternative="less")
                    sig_model_list.append(
                        {
                            "agg": agg_type,
                            "llama_model": llama_model,
                            "desc_seg": desc_seg,
                            "alpha2": alpha,
                            "p_val": p_val,
                            "stat": stat,
                            "avg_diff": round(np.mean(diff), 4),
                            "std_diff": round(np.std(diff), 4),
                            "n": len(diff),
                        }
                    )
                    # median = diff.median()
                    # diff.hist(bins=20)
                    # plt.axvline(median, linestyle='--',color='red')  # vertical line at median
                    # plt.show()
    sig_model_df = pd.DataFrame(sig_model_list)
    _, sig_model_df["p_val_correct"], _, _ = multipletests(
        pvals=sig_model_df["p_val"], method="fdr_bh"
    )
    return sig_model_df

In [ ]:
param_dict = {
    "agg": ["0_shot_binary", "0_shot_proba-limited", "avg", "block", "max"],
    "llama_model": ["Llama-3.3-8B-Instruct", "Llama-3.3-70B-Instruct"],
    "desc": ["desc_augment", "desc_augment_neg", "desc_original"],
    "seg": ["pt_noshort", "turns"],
    "alpha_2": ALPHA_LIST,
    "repeats": [f"repeat_{i}" for i in range(10)],
    "splits": [f"split_{i}.parquet" for i in range(3)],
}
self_hypothesis_5 = get_hypothesis_5(
    "self",
    param_dict=param_dict,
    base_dir=BASE_PATH / "preds" / "baseline",
)
display(self_hypothesis_5[self_hypothesis_5["p_val_correct"] < 0.05])

parent_hypothesis_5 = get_hypothesis_5(
    "parent",
    param_dict=param_dict,
    base_dir=BASE_PATH / "preds" / "baseline",
)
display(parent_hypothesis_5[parent_hypothesis_5["p_val_correct"] < 0.05])

In [ ]:
export_hypothesis(impact_type="self", table=self_hypothesis_5, hypothesis_num=5)
export_hypothesis(impact_type="parent", table=parent_hypothesis_5, hypothesis_num=5)

### Final intersect hypotheses

Parent

In [ ]:
import_path = BASE_PATH / "sig_tests"
parent_hypothesis_1 = pd.read_csv(
    import_path / "hypothesis_1/parent.tsv", sep="\t", index_col=0
)
parent_hypothesis_2 = pd.read_csv(
    import_path / "hypothesis_2/parent.tsv", sep="\t", index_col=0
)
parent_hypothesis_3 = pd.read_csv(
    import_path / "hypothesis_3/parent.tsv", sep="\t", index_col=0
)
parent_hypothesis_4 = pd.read_csv(
    import_path / "hypothesis_4/parent.tsv", sep="\t", index_col=0
)
parent_hypothesis_5 = pd.read_csv(
    import_path / "hypothesis_5/parent.tsv", sep="\t", index_col=0
)

In [ ]:
# Hypothesis 1
ph1_mcsig = parent_hypothesis_1.query("p_val_correct < 0.05")
ph1_mcsig["alpha2"] = ph1_mcsig["alpha"]
# Hypothesis 2
ph2_mcsig = parent_hypothesis_2.query("p_val_correct < 0.05")
ph12_mcsig = ph1_mcsig.merge(
    ph2_mcsig, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
)
# Hypothesis 3
ph123 = ph12_mcsig.merge(
    parent_hypothesis_3, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
).sort_values("p_val")
_, ph123["p_val_smaller_compare"], _, _ = multipletests(ph123.p_val, method="fdr_bh")
ph123_mcsig = ph123[ph123["p_val_smaller_compare"] < 0.05]
# Hypothesis 4
ph4_mcsig = parent_hypothesis_4.query("p_val_correct < 0.05")
ph4_mcsig["alpha2"] = ph4_mcsig["alpha"]
ph4_mcsig = ph4_mcsig.rename(
    columns={
        "p_val": "p_val_h4",
        "p_val_correct": "p_val_correct_h4",
        "stat": "stat_h4",
        "avg_diff": "avg_diff_h4",
    }
)
ph1234 = ph123_mcsig.merge(
    ph4_mcsig, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
)
# Hypothesis 5
ph5_mcsig = parent_hypothesis_5.query("p_val_correct < 0.05")
ph5_mcsig = ph5_mcsig.rename(
    columns={
        "p_val": "p_val_h5",
        "p_val_correct": "p_val_correct_h5",
        "stat": "stat_h5",
        "avg_diff": "avg_diff_h5",
    }
)
ph12345 = ph1234.merge(
    ph5_mcsig, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
)
ph12345

Self

In [ ]:
import_path = BASE_PATH / "sig_tests"
self_hypothesis_1 = pd.read_csv(
    import_path / "hypothesis_1/self.tsv", sep="\t", index_col=0
)
self_hypothesis_2 = pd.read_csv(
    import_path / "hypothesis_2/self.tsv", sep="\t", index_col=0
)
self_hypothesis_3 = pd.read_csv(
    import_path / "hypothesis_3/self.tsv", sep="\t", index_col=0
)
self_hypothesis_4 = pd.read_csv(
    import_path / "hypothesis_4/self.tsv", sep="\t", index_col=0
)
self_hypothesis_5 = pd.read_csv(
    import_path / "hypothesis_5/self.tsv", sep="\t", index_col=0
)

In [ ]:
self_hypothesis_5.sort_values("p_val")

In [ ]:
# Hypothesis 1
sh1_mcsig = self_hypothesis_1.query("p_val_correct < 0.05")
sh1_mcsig["alpha2"] = sh1_mcsig["alpha"]
# Hypothesis 2
sh2_mcsig = self_hypothesis_2.query("p_val_correct < 0.05")
sh12_mcsig = sh1_mcsig.merge(
    sh2_mcsig, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
)
# Hypothesis 3
sh123 = sh12_mcsig.merge(
    self_hypothesis_3, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
).sort_values("p_val")
_, sh123["p_val_smaller_compare"], _, _ = multipletests(sh123.p_val, method="fdr_bh")
sh123_mcsig = sh123[sh123["p_val_smaller_compare"] < 0.05]
# Hypothesis 4
sh4_mcsig = self_hypothesis_4.query("p_val_correct < 0.05")
sh4_mcsig["alpha2"] = sh4_mcsig["alpha"]
sh4_mcsig = sh4_mcsig.rename(
    columns={
        "p_val": "p_val_h4",
        "p_val_correct": "p_val_correct_h4",
        "stat": "stat_h4",
        "avg_diff": "avg_diff_h4",
    }
)
sh1234 = sh123_mcsig.merge(
    sh4_mcsig, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
)
# Hypothesis 5
sh5_mcsig = self_hypothesis_5.query("p_val_correct < 0.05")
self_hypothesis_5 = self_hypothesis_5.rename(
    columns={
        "p_val": "p_val_h5",
        "p_val_correct": "p_val_correct_h5",
        "stat": "stat_h5",
        "avg_diff": "avg_diff_h5",
    }
)
sh12345 = sh1234.merge(
    self_hypothesis_5, how="inner", on=["agg", "llama_model", "desc_seg", "alpha2"]
).sort_values("p_val")
sh12345 = (
    sh12345.sort_values(["agg", "llama_model", "desc_seg", "alpha2", "alpha3"])
    .groupby(["agg", "llama_model", "desc_seg", "alpha2"])
    .first()
    .reset_index()
)
_, sh12345["p_val_smaller_compare"], _, _ = multipletests(
    sh12345.p_val_h5, method="fdr_bh"
)
sh12345 = sh12345[sh12345["p_val_smaller_compare"] < 0.05]
sh12345

## Get avg r^2 for selected models

In [ ]:
parent_res_dict = {
    "1_layers": [],
    "2_layers": [],
    "3_layers": [],
}

In [ ]:
pd.read_parquet(
    "/data/MLDSST/mataray/DAWBA/preds/baseline/1_layers/ridge/0_shot_binary/parent/Llama-3.3-8B-Instruct/desc_augment_neg/alpha_5.0/repeat_1/split_0.parquet"
)

In [ ]:
pd.read_parquet(
    "/data/MLDSST/mataray/DAWBA/preds/baseline/2_layers/ridge/0_shot_binary/parent/Llama-3.3-8B-Instruct/desc_augment_neg/alpha_5.0/repeat_1/split_0.parquet"
)

In [ ]:
pd.read_parquet(
    "/data/MLDSST/mataray/DAWBA/preds/baseline/3_layers/ridge/0_shot_binary/parent/Llama-3.3-8B-Instruct/desc_augment_neg/alpha_5.0_0.5/repeat_1/split_0.parquet"
)

In [ ]:
(
    (parent_hypothesis_2["agg"] == "0_shot_proba-limited")
    & (parent_hypothesis_2["alpha2"] == 10.0)
    & (parent_hypothesis_2["llama_model"] == "Llama-3.3-8B-Instruct")
    & (parent_hypothesis_2["desc_seg"] == "desc_original")
    & (parent_hypothesis_2["llama_model"] == "Llama-3.3-8B-Instruct")
    & (parent_hypothesis_2["alpha3"] == 1.0)
)

import_dir = (
    BASE_PATH
    / "preds"
    / "baseline"
    / "3_layers"
    / "ridge"
    / "0_shot_proba-limited"
    / "parent"
    / "Llama-3.3-8B-Instruct"
    / "desc_original"
    / "alpha_10.0_1.0"
)
r2_dict = {
    "sum": [],
    "quest": [],
    "text": [],
}
rmse_dict = {
    "sum": [],
    "quest": [],
    "text": [],
}
for repeat_dir in import_dir.iterdir():
    dfs = []
    for split_path in repeat_dir.iterdir():
        dfs.append(pd.read_parquet(split_path))
    df_combined = pd.concat(dfs)
    y_true = df_combined["true"]

    y_pred1 = df_combined["pred_stage1"]
    r2_dict["sum"].append(r2_score(y_true, y_pred1))
    rmse_dict["sum"].append(root_mean_squared_error(y_true, y_pred1))

    y_pred2 = df_combined["pred_stage2_total"]
    r2_dict["quest"].append(r2_score(y_true, y_pred2))
    rmse_dict["quest"].append(root_mean_squared_error(y_true, y_pred2))

    y_pred3 = df_combined["pred_stage3_total"]
    r2_dict["text"].append(r2_score(y_true, y_pred3))
    rmse_dict["text"].append(root_mean_squared_error(y_true, y_pred3))
new_row = {
    "S_r2": np.mean(r2_dict["sum"]),
    "Q_r2": np.mean(r2_dict["quest"]),
    "T_r2": np.mean(r2_dict["text"]),
    "S_rmse": np.mean(rmse_dict["sum"]),
    "Q_rmse": np.mean(rmse_dict["quest"]),
    "T_rmse": np.mean(rmse_dict["text"]),
}
new_row

In [ ]:
import_dir = (
    BASE_PATH
    / "preds"
    / "baseline"
    / "3_layers"
    / "ridge"
    / "0_shot_proba-limited"
    / "parent"
    / "Llama-3.3-8B-Instruct"
    / "desc_original"
    / "alpha_10.0_1.0"
)
for repeat_dir in import_dir.iterdir():
    dfs = []
    for split_path in repeat_dir.iterdir():
        dfs.append(pd.read_parquet(split_path))
    df_combined = pd.concat(dfs)
    y_true = df_combined["true"]
    y_pred1 = df_combined["pred_stage1"]
    y_pred2 = df_combined["pred_stage2_total"]
    y_pred3 = df_combined["pred_stage3_total"]
    print(r2_score(y_true, y_pred2))
    print(r2_score(y_true, y_pred3))
    # plt.scatter(y_true, y_pred1, label='Sum', color='blue', marker='o', alpha=0.7)
    plt.scatter(
        y_true, y_pred2, label="Questionnaire", color="orange", marker="s", alpha=0.7
    )
    plt.scatter(y_true, y_pred3, label="Text", color="green", marker="^", alpha=0.7)

    # Perfect prediction reference line
    plt.plot(
        [min(y_true), max(y_true)],
        [min(y_true), max(y_true)],
        "k--",
        linewidth=1,
        label="Perfect fit",
    )

    plt.xlabel("y_true")
    plt.ylabel("y_pred")
    plt.title("True vs Predicted — All Models")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
for num_layers in parent_res_dict.keys():
    all_rows = []
    print(num_layers)
    sub_df = ph12345[["agg", "llama_model", "desc_seg", "alpha2", "alpha3"]]
    for _, row in sub_df.iterrows():
        base_dir = (
            BASE_PATH
            / "preds"
            / "baseline"
            / num_layers
            / "ridge"
            / row["agg"]
            / "parent"
            / row["llama_model"]
            / row["desc_seg"]
        )
        if num_layers in ["1_layers", "2_layers"]:
            ## Only use alpha2
            alpha_dir = base_dir / f"alpha_{row.alpha2}"
        elif num_layers == "3_layers":
            ## Use alpha2 + 3
            alpha_dir = base_dir / f"alpha_{row.alpha2}_{row.alpha3}"
        else:
            raise ValueError()
        r2_dict = {
            "sum": [],
            "quest": [],
            "text": [],
        }
        rmse_dict = {
            "sum": [],
            "quest": [],
            "text": [],
        }
        for repeat_dir in alpha_dir.iterdir():
            dfs = []
            for split_path in repeat_dir.iterdir():
                dfs.append(pd.read_parquet(split_path))
            df_combined = pd.concat(dfs)
            y_true = df_combined["true"]

            y_pred1 = df_combined["pred_stage1"]
            r2_dict["sum"].append(r2_score(y_true, y_pred1))
            rmse_dict["sum"].append(root_mean_squared_error(y_true, y_pred1))

            y_pred2 = df_combined["pred_stage2_total"]
            r2_dict["quest"].append(r2_score(y_true, y_pred2))
            rmse_dict["quest"].append(root_mean_squared_error(y_true, y_pred2))

            y_pred3 = df_combined["pred_stage3_total"]
            r2_dict["text"].append(r2_score(y_true, y_pred3))
            rmse_dict["text"].append(root_mean_squared_error(y_true, y_pred3))
        if num_layers in [1, 2]:
            row_dict = {k: v for k, v in row.items() if k != "alpha3"}
        else:
            row_dict = row
        new_row = {
            **row_dict,
            "S_r2": np.mean(r2_dict["sum"]),
            "Q_r2": np.mean(r2_dict["quest"]),
            "T_r2": np.mean(r2_dict["text"]),
            "S_rmse": np.mean(rmse_dict["sum"]),
            "Q_rmse": np.mean(rmse_dict["quest"]),
            "T_rmse": np.mean(rmse_dict["text"]),
        }
        all_rows.append(new_row)
    display(pd.DataFrame(all_rows))

In [ ]:
for num_layers in parent_res_dict.keys():
    all_rows = []
    print(num_layers)
    sub_df = sh12345[["agg", "llama_model", "desc_seg", "alpha2", "alpha3"]]
    for _, row in sub_df.iterrows():
        base_dir = (
            BASE_PATH
            / "preds"
            / "baseline"
            / num_layers
            / "ridge"
            / row["agg"]
            / "self"
            / row["llama_model"]
            / row["desc_seg"]
        )
        if num_layers in ["1_layers", "2_layers"]:
            ## Only use alpha2
            alpha_dir = base_dir / f"alpha_{row.alpha2}"
        elif num_layers == "3_layers":
            ## Use alpha2 + 3
            alpha_dir = base_dir / f"alpha_{row.alpha2}_{row.alpha3}"
        else:
            raise ValueError()
        r2_dict = {
            "sum": [],
            "quest": [],
            "text": [],
        }
        rmse_dict = {
            "sum": [],
            "quest": [],
            "text": [],
        }
        for repeat_dir in alpha_dir.iterdir():
            dfs = []
            for split_path in repeat_dir.iterdir():
                dfs.append(pd.read_parquet(split_path))
            df_combined = pd.concat(dfs)
            y_true = df_combined["true"]

            y_pred1 = df_combined["pred_stage1"]
            r2_dict["sum"].append(r2_score(y_true, y_pred1))
            rmse_dict["sum"].append(root_mean_squared_error(y_true, y_pred1))

            y_pred2 = df_combined["pred_stage2_total"]
            r2_dict["quest"].append(r2_score(y_true, y_pred2))
            rmse_dict["quest"].append(root_mean_squared_error(y_true, y_pred2))

            y_pred3 = df_combined["pred_stage3_total"]
            r2_dict["text"].append(r2_score(y_true, y_pred3))
            rmse_dict["text"].append(root_mean_squared_error(y_true, y_pred3))
        if num_layers in [1, 2]:
            row_dict = {k: v for k, v in row.items() if k != "alpha3"}
        else:
            row_dict = row
        new_row = {
            **row_dict,
            "S_r2": np.mean(r2_dict["sum"]),
            "Q_r2": np.mean(r2_dict["quest"]),
            "T_r2": np.mean(r2_dict["text"]),
            "S_rmse": np.mean(rmse_dict["sum"]),
            "Q_rmse": np.mean(rmse_dict["quest"]),
            "T_rmse": np.mean(rmse_dict["text"]),
        }
        all_rows.append(new_row)
    display(pd.DataFrame(all_rows))

## Secondary Analyses

### 3T vs 3S



In [ ]:
def get_hypothesis_2a(impact_type):
    """
    Given a three-tier modeling schema:
        H_0: text level <= sum level
        H_A: text level > sum level
    """
    loop_dir = BASE_PATH / "preds/baseline/3_layers/ridge"
    sig_model_list = []
    for agg_dir in loop_dir.iterdir():
        agg_name = agg_dir.name
        impact_dir = agg_dir / impact_type
        for llama_dir in impact_dir.iterdir():
            llama_name = llama_dir.name
            for desc_seg_dir in llama_dir.iterdir():
                desc_seg_name = desc_seg_dir.name
                for alpha_dir in desc_seg_dir.iterdir():
                    alpha_dir_split = alpha_dir.name.split("_")
                    alpha2 = float(alpha_dir_split[1])
                    alpha3 = float(alpha_dir_split[2])
                    ##### At this point, we have all the unique params of this model
                    ## Now aggregate folds to get RSE for each participant in a given repeat, and avg over all repeats
                    ## End with avg RSE for each participant over repeats
                    repeat_dfs = []
                    for repeat_dir in alpha_dir.iterdir():
                        split_dfs = []
                        for split_path in repeat_dir.iterdir():
                            split_file = pd.read_parquet(split_path)
                            split_dfs.append(split_file)
                        combined_splits_df = pd.concat(split_dfs)
                        combined_splits_df["rse_3S"] = root_squared_error(
                            y_true=combined_splits_df["true"],
                            y_pred=combined_splits_df["pred_stage1"],
                        )
                        combined_splits_df["rse_3T"] = root_squared_error(
                            y_true=combined_splits_df["true"],
                            y_pred=combined_splits_df["pred_stage3_total"],
                        )
                        repeat_dfs.append(
                            combined_splits_df[
                                [
                                    "DAWBA_ID",
                                    # "true",
                                    # "pred_stage1",
                                    # "pred_stage3_total",
                                    "rse_3S",
                                    "rse_3T",
                                ]
                            ].copy()
                        )
                    all_preds = pd.concat(repeat_dfs, ignore_index=True)
                    # avg RSE over n repeats for each participant
                    participant_avg = all_preds.groupby("DAWBA_ID", as_index=False).agg(
                        mrse_3S=("rse_3S", "mean"),
                        mrse_3T=("rse_3T", "mean"),
                        n_repeats=("rse_3S", "size"),
                    )
                    assert (
                        participant_avg["n_repeats"] == 10
                    ).all(), "Some participants have missing repeats!"
                    diff = participant_avg["mrse_3T"] - participant_avg["mrse_3S"]
                    stat, p_val = wilcoxon(x=diff, alternative="less")
                    sig_model_list.append(
                        {
                            "agg": agg_name,
                            "llama_model": llama_name,
                            "desc_seg": desc_seg_name,
                            "alpha2": alpha2,
                            "alpha3": alpha3,
                            "p_val": p_val,
                            "stat": stat,
                            "avg_diff": round(np.mean(diff), 4),
                        }
                    )
                    # median = diff.median()
                    # diff.hist(bins=20)
                    # plt.axvline(median, linestyle='--',color='red')  # vertical line at median
                    # plt.show()
    sig_model_df = pd.DataFrame(sig_model_list)
    _, sig_model_df["p_val_correct"], _, _ = multipletests(
        pvals=sig_model_df["p_val"], method="fdr_bh"
    )
    return sig_model_df

In [ ]:
self_hypothesis_2a = get_hypothesis_2a("self")
display(self_hypothesis_2a[self_hypothesis_2a["p_val_correct"] < 0.05])

parent_hypothesis_2a = get_hypothesis_2a("parent")
display(parent_hypothesis_2a[parent_hypothesis_2a["p_val_correct"] < 0.05])

In [ ]:
parent_hypothesis_2a.sort_values("p_val")

### 3Q vs 3S

In [ ]:
def get_hypothesis_2b(impact_type):
    """
    Given a three-tier modeling schema:
        H_0: text level <= sum level
        H_A: text level > sum level
    """
    loop_dir = BASE_PATH / "preds/baseline/3_layers/ridge"
    sig_model_list = []
    for agg_dir in loop_dir.iterdir():
        agg_name = agg_dir.name
        impact_dir = agg_dir / impact_type
        for llama_dir in impact_dir.iterdir():
            llama_name = llama_dir.name
            for desc_seg_dir in llama_dir.iterdir():
                desc_seg_name = desc_seg_dir.name
                for alpha_dir in desc_seg_dir.iterdir():
                    alpha_dir_split = alpha_dir.name.split("_")
                    alpha2 = float(alpha_dir_split[1])
                    alpha3 = float(alpha_dir_split[2])
                    ##### At this point, we have all the unique params of this model
                    ## Now aggregate folds to get RSE for each participant in a given repeat, and avg over all repeats
                    ## End with avg RSE for each participant over repeats
                    repeat_dfs = []
                    for repeat_dir in alpha_dir.iterdir():
                        split_dfs = []
                        for split_path in repeat_dir.iterdir():
                            split_file = pd.read_parquet(split_path)
                            split_dfs.append(split_file)
                        combined_splits_df = pd.concat(split_dfs)
                        combined_splits_df["rse_3S"] = root_squared_error(
                            y_true=combined_splits_df["true"],
                            y_pred=combined_splits_df["pred_stage1"],
                        )
                        combined_splits_df["rse_3Q"] = root_squared_error(
                            y_true=combined_splits_df["true"],
                            y_pred=combined_splits_df["pred_stage2_total"],
                        )
                        repeat_dfs.append(
                            combined_splits_df[
                                [
                                    "DAWBA_ID",
                                    # "true",
                                    # "pred_stage1",
                                    # "pred_stage3_total",
                                    "rse_3S",
                                    "rse_3Q",
                                ]
                            ].copy()
                        )
                    all_preds = pd.concat(repeat_dfs, ignore_index=True)
                    # avg RSE over n repeats for each participant
                    participant_avg = all_preds.groupby("DAWBA_ID", as_index=False).agg(
                        mrse_3S=("rse_3S", "mean"),
                        mrse_3Q=("rse_3Q", "mean"),
                        n_repeats=("rse_3S", "size"),
                    )
                    assert (
                        participant_avg["n_repeats"] == 10
                    ).all(), "Some participants have missing repeats!"
                    diff = participant_avg["mrse_3Q"] - participant_avg["mrse_3S"]
                    stat, p_val = wilcoxon(x=diff, alternative="greater")
                    sig_model_list.append(
                        {
                            "agg": agg_name,
                            "llama_model": llama_name,
                            "desc_seg": desc_seg_name,
                            "alpha2": alpha2,
                            "alpha3": alpha3,
                            "p_val": p_val,
                            "stat": stat,
                            "avg_diff": round(np.mean(diff), 4),
                        }
                    )
                    # median = diff.median()
                    # diff.hist(bins=20)
                    # plt.axvline(median, linestyle='--',color='red')  # vertical line at median
                    # plt.show()
    sig_model_df = pd.DataFrame(sig_model_list)
    _, sig_model_df["p_val_correct"], _, _ = multipletests(
        pvals=sig_model_df["p_val"], method="fdr_bh"
    )
    return sig_model_df

In [ ]:
self_hypothesis_2b = get_hypothesis_2b("self")
display(self_hypothesis_2b[self_hypothesis_2b["p_val_correct"] < 0.05])

parent_hypothesis_2b = get_hypothesis_2b("parent")
display(parent_hypothesis_2b[parent_hypothesis_2b["p_val_correct"] < 0.05])

### 2T vs 3T


In [ ]:
def get_hypothesis_3a(impact_type, param_dict, base_dir):
    """
    Given a two and three-tier modeling schema:
        H_0: 2-level text == 3-level text
        H_A: 2-level text != 3-level text
    """
    # loop_dir = BASE_PATH / "preds/baseline/3_layers/ridge"
    sig_model_list = []
    for agg_type in param_dict["agg"]:
        for llama_model in param_dict["llama_model"]:
            if agg_type in ["avg", "max", "block"]:
                desc_seg_list = param_dict["seg"]
            else:
                desc_seg_list = param_dict["desc"]
            for desc_seg in desc_seg_list:
                for alpha in param_dict["alpha_2"]:
                    alpha_dir_2 = (
                        base_dir
                        / "2_layers/ridge"
                        / agg_type
                        / impact_type
                        / llama_model
                        / desc_seg
                        / f"alpha_{alpha}"
                    )
                    # use arbitrary val for second value of alpha, since only first is used in questionnaire level
                    alpha_dir_3 = (
                        base_dir
                        / "3_layers/ridge"
                        / agg_type
                        / impact_type
                        / llama_model
                        / desc_seg
                        / f"alpha_{alpha}_0.5"
                    )
                    ##### At this point, we have all the unique params of this model
                    ## Now aggregate folds to get RSE for each participant in a given repeat, and avg over all repeats
                    ## End with avg RSE for each participant over repeats
                    repeat_dfs = []
                    for repeat in param_dict["repeats"]:
                        split_dfs_2 = []
                        split_dfs_3 = []
                        for split_name in param_dict["splits"]:
                            split_path_2 = alpha_dir_2 / repeat / split_name
                            split_path_3 = alpha_dir_3 / repeat / split_name
                            split_file_2 = pd.read_parquet(split_path_2)
                            split_file_3 = pd.read_parquet(split_path_3)
                            split_dfs_2.append(split_file_2)
                            split_dfs_3.append(split_file_3)
                        combined_splits_df_2 = pd.concat(split_dfs_2)
                        combined_splits_df_3 = pd.concat(split_dfs_3)
                        combined_splits_df_2["rse_2T"] = root_squared_error(
                            y_true=combined_splits_df_2["true"],
                            y_pred=combined_splits_df_2["pred_stage3_total"],
                        )
                        combined_splits_df_3["rse_3T"] = root_squared_error(
                            y_true=combined_splits_df_3["true"],
                            y_pred=combined_splits_df_3["pred_stage3_total"],
                        )
                        combined_all = pd.merge(
                            combined_splits_df_2[["DAWBA_ID", "rse_2T"]],
                            combined_splits_df_3[["DAWBA_ID", "rse_3T"]],
                            on="DAWBA_ID",
                        )
                        repeat_dfs.append(combined_all.copy())
                    all_preds = pd.concat(repeat_dfs, ignore_index=True)
                    # avg RSE over n repeats for each participant
                    participant_avg = all_preds.groupby("DAWBA_ID", as_index=False).agg(
                        mrse_3T=("rse_3T", "mean"),
                        mrse_2T=("rse_2T", "mean"),
                        n_repeats=("rse_3T", "size"),
                    )
                    assert (
                        participant_avg["n_repeats"] == 10
                    ).all(), "Some participants have missing repeats!"
                    diff = participant_avg["mrse_3T"] - participant_avg["mrse_2T"]
                    stat, p_val = wilcoxon(x=diff, alternative="greater")
                    sig_model_list.append(
                        {
                            "agg": agg_type,
                            "llama_model": llama_model,
                            "desc_seg": desc_seg,
                            "alpha2": alpha,
                            "p_val": p_val,
                            "stat": stat,
                            "avg_diff": round(np.mean(diff), 4),
                        }
                    )
                    # median = diff.median()
                    # diff.hist(bins=20)
                    # plt.axvline(median, linestyle='--',color='red')  # vertical line at median
                    # plt.show()
    sig_model_df = pd.DataFrame(sig_model_list)
    _, sig_model_df["p_val_correct"], _, _ = multipletests(
        pvals=sig_model_df["p_val"], method="fdr_bh"
    )
    return sig_model_df

In [ ]:
self_hypothesis_3a = get_hypothesis_3a(
    impact_type="self", param_dict=param_dict, base_dir=BASE_PATH / "preds" / "baseline"
)
display(self_hypothesis_3a[self_hypothesis_3a["p_val_correct"] < 0.05])

parent_hypothesis_3a = get_hypothesis_3a(
    impact_type="parent",
    param_dict=param_dict,
    base_dir=BASE_PATH / "preds" / "baseline",
)
display(parent_hypothesis_3a[parent_hypothesis_3a["p_val_correct"] < 0.05])